
### Objective:
1. Verify that VMVs are mapped to parameter type codes in **REF_SamplingParams**, if so collect their **Parameter short name**
2. Match the **Parameter short name** to its **Name** in **REF_ParameterLists**, then place in a set
3. **Make sure you know if you've got STAR or VMV codes! And to use up to date mappings!**

In [1]:
# environment testing - check working directory and pandas import
import sys, pandas as pd
print(sys.executable)

c:\Users\WilsonE\OneDrive - EC-EC\Main_Working\Random_Items\PythonWork\All_Purpose_VENV\Scripts\python.exe


In [2]:
# read files, load in memory
sp_ref = pd.read_csv("mappings/REF_2026-07-02_SamplingParameters_Export.csv")
pl_ref = pd.read_csv("mappings/REF_2026-07-02_ParameterLists_Export.csv")
star_ref = pd.read_csv("mappings/REF_2026-04-14_STAR_Mapping.csv")
file_name = input("Paste name of target file in root folder; e.g. = 'star_target.txt'")
target = pd.read_csv(file_name)

In [3]:
target.head(3)

,VMV_Code
0,79
1,80
2,201


In [4]:
# verify parameter list mappings are 1:1 -> output should be 0
pl_test = pl_ref.groupby("Parameter short name")["Name"].nunique()
dup_check = pl_test[pl_test > 1]
print(len(dup_check))

0


In [5]:
# remove duplicates from parameter list reference to prevent one to many merges
# This necessitates the above cell returning 0!
pl_ref = pl_ref.drop_duplicates(subset=["Parameter short name"])

In [6]:
# If STAR codes, merge target star codes with their mapped VMVs
is_star = input("Is this a list of STAR codes? 'Yes' or 'No'")
if is_star.lower() == "yes":
    target_merge = target.merge(
        star_ref[['CODE', 'National_VMV_Code']], 
        on='CODE', 
        how='left')
    target_merge["National_VMV_Code"] = target_merge["National_VMV_Code"].astype(int)
elif is_star.lower() == "no":
    current_column_name = target.columns[0]
    target_merge = target.rename(columns={current_column_name: "National_VMV_Code"})
else:
    print("Please answer 'Yes' or 'No'")

In [7]:
# check for any STAR codes not mapped to VMV codes. check_na should = 0
check_na = target_merge[target_merge.isna().any(axis=1)]
print(len(target_merge), len(check_na))

915 0


In [8]:
target_merge.head(3)

,National_VMV_Code
0,79
1,80
2,201


In [9]:
# merge the DMS parameter codes. 
# Use left_on and right_on instead of on and how because different column names.
target_merge = target_merge.merge(
    sp_ref[["Hydstra code", "Name", "Long name"]], 
    left_on='National_VMV_Code', 
    right_on='Hydstra code',
    how="left")
target_merge = target_merge.drop(columns=['Hydstra code'])

In [10]:
# check for VMVs not mapped to DMS parameter type codes
check_na = target_merge[target_merge.isna().any(axis=1)]
print(len(target_merge), len(check_na))
if len(check_na) > 0:
    print("The following VMV codes are not mapped to DMS parameter type codes:")
    print(check_na["National_VMV_Code"])

915 0


In [ ]:
# should look normal
target_merge.head(3)

,National_VMV_Code,Name,Long name
0,79,4541_00_00_00_14,Chlorophyll corrected
1,80,414_00_00_00_14,Chlorophyll A
2,201,837_00_00_00_14,Total dissolved solids (filterable residue) (c...


In [12]:
# merge parameter list values, add column for unique parameter list values
target_merge = target_merge.merge(
    pl_ref[["Name", "Parameter short name"]],
    left_on="Name",
    right_on="Parameter short name",
    how="left",)
target_merge.drop(columns=["Parameter short name"], inplace=True)
target_merge.rename(columns={"Name_x": "Parameter type code", "Name_y": "Parameter list value"}, inplace=True)
unique_vals = sorted(target_merge["Parameter list value"].dropna().unique())
target_merge["Unique parameter list values"] = pd.Series(unique_vals)

In [13]:
# check for NA again
check_na = target_merge[target_merge.iloc[:, :4].isna().any(axis=1)]
print(len(check_na))
if len(check_na) > 0:
    print("Rows with NA values found:")
    print(check_na[["National_VMV_Code", "Parameter type code", "Parameter list value"]])

# toggle for receipt - should be len 0
check_na.to_csv(f"{file_name.rsplit('.',1)[0]}_NA_check.csv", index=False)

0


In [14]:
target_merge.head(3)

,National_VMV_Code,Parameter type code,Long name,Parameter list value,Unique parameter list values
0,79,4541_00_00_00_14,Chlorophyll corrected,Water - Biological,Sediment - Metal
1,80,414_00_00_00_14,Chlorophyll A,Water - Biological,Water - Biological
2,201,837_00_00_00_14,Total dissolved solids (filterable residue) (c...,Water - Physical,Water - Carbamate


In [15]:
# generate list of unique parameter list values
pl_list = target_merge["Parameter list value"].unique()
print(len(pl_list))
for x in sorted(pl_list):
    print(x)

17
Sediment - Metal
Water - Biological
Water - Carbamate
Water - Carbon
Water - Carbonated compounds
Water - Descriptive
Water - Hydrocarbons
Water - Major Ions
Water - Metal
Water - Neonicotinoid
Water - Nonmetals
Water - Nutrients
Water - Oxygen
Water - Pesticides
Water - Physical
Water - Sulfonyl urea
Water - Surrogate


In [16]:
target_merge.to_csv(f"{file_name.rsplit('.',1)[0]}_output.csv", index=False)